<a href="https://colab.research.google.com/github/Honorine-Kabore/genomic_mosq/blob/main/diversity_heterozygosity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q --no-warn-conflicts malariagen_data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 19.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.8/215.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 M

In [2]:
import malariagen_data
import allel
import numpy as np
import plotly.express as px
import plotly.io as pio
import pandas as pd

In [ ]:
# plotting setup
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
import matplotlib_venn as venn

In [3]:
#Mounting Google Drive
import os
from google.colab import drive
drive.mount("drive")

# make dir
results_dir = "drive/MyDrive/Genomic/"
os.makedirs(results_dir, exist_ok=True)

Mounted at drive


In [4]:
ag3 = malariagen_data.Ag3(pre=True, results_cache=results_dir)
ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
Results cache                         : /content/drive/MyDrive/Genomic
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.6.0
Client location                       : South Carolina, United States (Google Cloud us-east1)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
---
Please note that data are subject to terms of use,
for more information see https://www.malariagen.net/data
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.6.0/Ag3.html

In [5]:
sets =['3.7', '3.8','3.9', '3.10','3.11']
df_samples = ag3.sample_metadata(sample_sets=sets)
sample_query='country=="Burkina Faso"'


In [ ]:
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')

In [6]:
# Create a new cohort for taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_ta = {}
for ta in bf_samples.taxon.unique():
 for lo in bf_samples.location.unique():
  sample_list = list(bf_samples.query("taxon==@ta").sample_id)
  cohort_ta['An.'+ta] = "sample_id in ['"+ "', '".join(sample_list) + "']"

In [7]:
#create a new cohort with location, taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_re = {}
for ta in bf_samples.taxon.unique():
    for re in bf_samples.admin1_name.unique():
        sample_list = list(bf_samples.query("admin1_name==@re and taxon==@ta").sample_id)
        cohort_re[re + "_" + ta[:4]] = "sample_id in ['"+ "', '".join(sample_list) + "']"

# Diversity

In [9]:
df_stat = ag3.diversity_stats(
    sample_sets=sets,
    sample_query="country=='Burkina Faso'",
    cohorts=cohort_re,
    cohort_size=10,
    region="3L:15,000,000-41,000,000",
    site_mask="gamb_colu_arab",
    site_class="CDS_DEG_4"
)
df_stat

Cohort (Boucle du Mouhoun_gamb) has insufficient samples (2) for requested cohort size (10), dropping.
Cohort (Sahel_gamb) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Plateau Central_gamb) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Centre-Ouest_gamb) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Plateau Central_colu) has insufficient samples (3) for requested cohort size (10), dropping.
Cohort (Centre-Ouest_colu) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Cascades_arab) has insufficient samples (8) for requested cohort size (10), dropping.
Cohort (Plateau Central_arab) has insufficient samples (4) for requested cohort size (10), dropping.
Cohort (Centre-Ouest_arab) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Hauts-Bassins_unas) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Boucle d

,cohort,theta_pi,theta_pi_estimate,theta_pi_bias,theta_pi_std_err,theta_pi_ci_err,theta_pi_ci_low,theta_pi_ci_upp,theta_w,theta_w_estimate,...,tajima_d_ci_upp,taxon,year,month,country,admin1_iso,admin1_name,admin2_name,longitude,latitude
0,Hauts-Bassins_gamb,0.023331,0.023338,-6.839909e-06,0.000319,0.001252,0.022712,0.023963,0.036252,0.036259,...,-1.466809,gambiae,"[2012, 2017, 2018, 2019, 2020, 2021, 2022]","[-1, 7, 8, 9, 10, 11, 12]",Burkina Faso,BF-09,Hauts-Bassins,Houet,-4.432553,11.208211
1,Cascades_gamb,0.021764,0.021767,-3.168743e-06,0.000310,0.001216,0.021159,0.022375,0.031905,0.031905,...,-1.303474,gambiae,2022,10,Burkina Faso,BF-02,Cascades,Comoe,-4.267290,10.683323
2,Centre-Sud_gamb,0.021721,0.021726,-4.926708e-06,0.000315,0.001233,0.021110,0.022343,0.032135,0.032137,...,-1.326708,gambiae,2022,"[10, 11]",Burkina Faso,BF-07,Centre-Sud,Nahouri,-1.020000,11.219000
3,Est_gamb,0.021885,0.021891,-6.671272e-06,0.000310,0.001217,0.021283,0.022500,0.032470,0.032476,...,-1.339075,gambiae,2022,"[10, 11]",Burkina Faso,BF-08,Est,"[Gnagna, Tapoa]",1.143200,12.303300
4,Hauts-Bassins_colu,0.022505,0.022511,-5.642596e-06,0.000320,0.001256,0.021883,0.023139,0.032685,0.032694,...,-1.277275,coluzzii,"[2012, 2017, 2018, 2019, 2020, 2021, 2022]","[-1, 3, 6, 7, 8, 9, 10, 11, 12]",Burkina Faso,BF-09,Hauts-Bassins,Houet,-4.486339,11.231664
5,Boucle du Mouhoun_colu,0.022362,0.022368,-5.654440e-06,0.000327,0.001282,0.021727,0.023009,0.032472,0.032477,...,-1.276565,coluzzii,"[2008, 2022]","[9, 11]",Burkina Faso,BF-01,Boucle du Mouhoun,"[Kossi, Sourou]",-3.079889,12.971556
6,Cascades_colu,0.022860,0.022865,-5.165276e-06,0.000317,0.001244,0.022243,0.023487,0.033677,0.033682,...,-1.318081,coluzzii,"[2011, 2012, 2015, 2016, 2022]","[-1, 10]",Burkina Faso,BF-02,Cascades,Comoe,-4.759470,10.699028
7,Centre-Sud_colu,0.022221,0.022227,-6.103490e-06,0.000311,0.001218,0.021618,0.022836,0.032150,0.032157,...,-1.265558,coluzzii,2022,"[10, 11]",Burkina Faso,BF-07,Centre-Sud,Nahouri,-1.020000,11.219000
8,Est_colu,0.021826,0.021831,-4.376195e-06,0.000316,0.001238,0.021212,0.022450,0.031263,0.031268,...,-1.236861,coluzzii,2022,"[10, 11]",Burkina Faso,BF-08,Est,"[Gnagna, Tapoa]",0.392216,12.667748
9,Sahel_colu,0.021964,0.021968,-4.499816e-06,0.000317,0.001242,0.021347,0.022589,0.031319,0.031321,...,-1.224850,coluzzii,2022,10,Burkina Faso,BF-12,Sahel,Oudalan,-0.128000,14.375000


In [10]:
fig = ag3.plot_diversity_stats(df_stat, color="taxon", show=True)

In [11]:
figs = ag3.plot_diversity_stats(df_stat, color="taxon", show=False)

In [12]:
figs[0]

In [13]:
figs[1]

In [14]:
figs[2]

In [15]:
figs[3]

In [ ]:
#create a new cohort with region and taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_re_ye = {}
for ta in bf_samples.taxon.unique():
    for re in bf_samples.admin1_name.unique():
      for ye in bf_samples.year.unique():
        sample_list = list(bf_samples.query("admin1_name==@re and taxon==@ta and year==@ye").sample_id)
        cohort_re_ye[re + "_" + ta[:4]+ '_' + str(ye)] = "sample_id in ['"+ "', '".join(sample_list) + "']"

In [ ]:
#region, taxon, year
df_stats_region_year= ag3.diversity_stats(
    sample_sets=sets,
    #sample_query="country=='Burkina Faso'",
    cohorts=cohort_re_ye,
    cohort_size=10,
    region="3L:15,000,000-41,000,000",
    site_mask="gamb_colu_arab",
    site_class="CDS_DEG_4"
)
df_stats_region_year

In [ ]:
fig = ag3.plot_diversity_stats(df_stats_region_year, color="taxon", show=True)

In [ ]:
figs = ag3.plot_diversity_stats(df_stats_region_year, color="taxon", show=False)

In [ ]:
figs[0]

In [ ]:
figs[1]

In [ ]:
figs[2]

In [ ]:
figs[3]

# Heterozygosity

In [ ]:
#est colu
ag3.plot_roh(
    sample="VBS93782-7551STDY14568441",
    region="3R",
    site_mask="gamb_colu",
    window_size=10_000,
)

In [ ]:
#coluzzii 2022 gnagna Est
ag3.plot_roh(
    sample="VBS93786-7551STDY14568445",
    region="3R",
    site_mask="gamb_colu",
    window_size=10_000,)

Compute heterozygous genotypes:   0%|          | 0/351 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/malariagen_data/anopheles.py:333: BokehDeprecationWarning:

'circle() method with size value' was deprecated in Bokeh 3.4.0 and will be removed, use 'scatter(size=...) instead' instead.



In [ ]:
#colu houet 2022
ag3.plot_roh(
    sample="VBS92807-7551STDY14567455",
    region="3R",
    site_mask="gamb_colu",
    window_size=10_000,
)

Compute heterozygous genotypes:   0%|          | 0/351 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/malariagen_data/anopheles.py:333: BokehDeprecationWarning:

'circle() method with size value' was deprecated in Bokeh 3.4.0 and will be removed, use 'scatter(size=...) instead' instead.



In [ ]:
##Arabiensis Sourgou 2022 houet
ag3.plot_roh(
    sample="VBS93285-7551STDY14567938",
    region="3L",
    site_mask="gamb_colu_arab",
    window_size=10_000,
)

In [ ]:
#arabiensis  nagare
ag3.plot_roh(
    sample="VBS93763-7551STDY14568422",
    region="3R",
    site_mask="gamb_colu_arab",
    window_size=10_000,
)

In [ ]:
##Arabiensis Sourgou 2022 houet
ag3.plot_roh(
    sample="VBS93290-7551STDY14567943",
    region="3R",
    site_mask="gamb_colu_arab",
    window_size=10_000,
)

Compute heterozygous genotypes:   0%|          | 0/351 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/malariagen_data/anopheles.py:333: BokehDeprecationWarning:

'circle() method with size value' was deprecated in Bokeh 3.4.0 and will be removed, use 'scatter(size=...) instead' instead.



In [ ]:
#arabiensis  cascades 2022
ag3.plot_roh(
    sample="VBS93408-7551STDY14568064",
    region="3R",
    site_mask="gamb_colu_arab",
    window_size=10_000,
)

In [ ]:
#gambiae
ag3.plot_roh(
    sample="VBS93037-7551STDY14567688",
    region="3R",
    site_mask="gamb_colu",
    window_size=10_000,
)

Compute heterozygous genotypes:   0%|          | 0/351 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/malariagen_data/anopheles.py:333: BokehDeprecationWarning:

'circle() method with size value' was deprecated in Bokeh 3.4.0 and will be removed, use 'scatter(size=...) instead' instead.

